# Reproducing the measurements of the lightweight cipher survey

This notebook re-runs the code behind the empirical section of the survey on **your** machine and
reports, value by value, what matches the stored results.

- **Structural results** (avalanche, SAC, S-box tables, statistical tests, key schedule diffusion,
  image metrics) use fixed random seeds and should match exactly.
- **Timing results** depend on your machine and are expected to differ. Report your own.
- **Code size and cycle counts** need the AVR and ARM cross-compilers (Linux or Google Colab).

Run the cells in order. The whole notebook takes about five minutes. The last cell prints the
platform sentence to put in Section 9.13 of the paper.

## 1. Get the code

In [ ]:
import os, sys, subprocess, zipfile, shutil
REPO = "lwc-survey-benchmark"
if not os.path.isdir(REPO):
    if os.path.exists(REPO + ".zip"):
        zipfile.ZipFile(REPO + ".zip").extractall(".")
    else:
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/ujjalmca/Lightweight-cipher-survey", REPO], check=True)
ROOT = os.path.abspath(REPO)
# when the notebook sits inside the repository itself
if os.path.exists("tests/test_vectors.py"):
    ROOT = os.path.abspath(".")
print("repository:", ROOT)
if "16-bit LCG" in open(os.path.join(ROOT, "software", "experiments3.py")).read():
    print("WARNING: this copy has the old statistical control in experiments3.py. "
          "Use the updated repository zip, otherwise the control result will not match.")
req = [sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(ROOT, "requirements.txt")]
if subprocess.run(req, capture_output=True).returncode != 0:
    subprocess.run(req + ["--break-system-packages"], capture_output=True)
print("Python packages ready")

## 2. Record the platform

In [ ]:
import platform, json, time
try:
    import cpuinfo; cpu = cpuinfo.get_cpu_info().get("brand_raw", "")
except Exception:
    cpu = platform.processor() or ""
    try:
        for line in open("/proc/cpuinfo"):
            if line.startswith("model name"):
                cpu = line.split(":", 1)[1].strip(); break
    except Exception:
        pass
gcc = subprocess.run(["gcc", "--version"], capture_output=True, text=True).stdout.splitlines()[:1]
PLATFORM = dict(os=platform.platform(), python=platform.python_version(), cpu=cpu,
                gcc=gcc[0] if gcc else "not found")
for k, v in PLATFORM.items(): print(f"{k:7s} {v}")

## 3. Keep a copy of the stored results
The scripts overwrite their result files, so the published values are saved first.

In [ ]:
RESULT_FILES = ["software/bench_results.json", "software/results2.json", "software/results3.json",
                "software/results4.json", "software/results_c.json",
                "hardware/exp_new_avalanche.json", "hardware/exp_new_stats.json",
                "hardware/exp_new_rounds.json", "hardware/exp_new_diffdecay.json"]
REF = os.path.join(ROOT, "_stored_results")
if not os.path.isdir(REF):
    for f in RESULT_FILES:
        dst = os.path.join(REF, f); os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy(os.path.join(ROOT, f), dst)
print("stored results saved in", REF)

## 4. Known-answer tests
Every cipher is checked against its test vector, in Python and in C.

In [ ]:
r = subprocess.run([sys.executable, "tests/test_vectors.py"], cwd=ROOT, capture_output=True, text=True)
print(r.stdout, r.stderr[-300:])
TESTS_OK = r.returncode == 0
if shutil.which("gcc"):
    exe = os.path.join(ROOT, "kat_c")
    subprocess.run(["gcc", "-O2", "-DTEST_MAIN", "-o", exe, "hardware/lwc_new.c"], cwd=ROOT)
    print(subprocess.run([exe], capture_output=True, text=True).stdout)
print("ALL PYTHON TESTS PASS" if TESTS_OK else "A TEST FAILED: stop here and report it")

## 5. Run the experiments
Seven scripts, about three to four minutes in total.

In [ ]:
SCRIPTS = [("software", "benchmark.py"), ("software", "experiments2.py"), ("software", "experiments3.py"),
           ("software", "experiments4.py"), ("hardware", "exp_new.py"), ("hardware", "exp_new2.py"),
           ("hardware", "exp_new3.py")]
for d, s in SCRIPTS:
    t0 = time.time()
    r = subprocess.run([sys.executable, s], cwd=os.path.join(ROOT, d), capture_output=True, text=True)
    print(f"{d}/{s:18s} {'ok' if r.returncode == 0 else 'FAILED'}  {time.time() - t0:5.0f} s")
    if r.returncode: print(r.stderr[-600:])

## 6. Native C benchmark
Compiles `software/lwc.c` and measures time per block on this machine.

In [ ]:
C_NEW = {}
if shutil.which("gcc"):
    exe = os.path.join(ROOT, "lwc_bench")
    subprocess.run(["gcc", "-O2", "-o", exe, "software/lwc.c"], cwd=ROOT)
    out = subprocess.run([exe], capture_output=True, text=True).stdout
    for line in out.split("BENCH_START")[-1].strip().splitlines():
        n, ns, cyc, mbit, ram = line.split()
        C_NEW[n] = dict(ns_blk=float(ns), cyc_byte=float(cyc), mbit=float(mbit))
    old = json.load(open(os.path.join(REF, "software/results_c.json")))
    print(f"{'cipher':13s}{'ns/block here':>15s}{'stored':>10s}{'cycles/byte here':>18s}{'stored':>10s}")
    for n, v in C_NEW.items():
        print(f"{n:13s}{v['ns_blk']:15.1f}{old[n]['ns_blk']:10.1f}{v['cyc_byte']:18.1f}{old[n]['cyc_byte']:10.1f}")
else:
    print("gcc not found: C benchmark skipped")

## 7. Compare with the stored results
Timing fields are listed separately, since they are expected to change.

In [ ]:
def flat(d, p=""):
    o = {}
    if isinstance(d, dict):
        for k, v in d.items(): o.update(flat(v, p + "/" + str(k)))
    elif isinstance(d, list):
        for i, v in enumerate(d): o.update(flat(v, p + "/" + str(i)))
    else: o[p] = d
    return o
TIMING = ("us_per_block", "kbps", "platform", "python", "ns_blk", "cyc_byte", "mbit")
summary, timing_rows, mismatches = [], [], []
for f in RESULT_FILES:
    if f.endswith("results_c.json"): continue
    a = flat(json.load(open(os.path.join(REF, f)))); b = flat(json.load(open(os.path.join(ROOT, f))))
    same = diff = tim = 0
    for k in a:
        if k not in b: continue
        if k.split("/")[-1] in TIMING:
            tim += 1
            if isinstance(a[k], (int, float)): timing_rows.append((f, k, a[k], b[k]))
        elif a[k] == b[k]: same += 1
        else: diff += 1; mismatches.append((f, k, a[k], b[k]))
    summary.append((f, same, diff, tim))
print(f"{'result file':36s}{'identical':>10s}{'different':>10s}{'timing':>8s}")
for f, s, d, t in summary: print(f"{f:36s}{s:10d}{d:10d}{t:8d}")
print()
if mismatches:
    print("NON-TIMING VALUES THAT DIFFER (these need attention):")
    for m in mismatches: print("  ", m)
else:
    print("Every non-timing value is identical to the stored result.")

In [ ]:
print("Python timing on this machine against the stored values (Table 7 of the paper):\n")
print(f"{'cipher':13s}{'microsec/block here':>20s}{'stored':>10s}{'kbit/s here':>14s}{'stored':>10s}")
new = json.load(open(os.path.join(ROOT, "software/bench_results.json")))["results"]
old = json.load(open(os.path.join(REF, "software/bench_results.json")))["results"]
for n in new:
    print(f"{n:13s}{new[n]['us_per_block']:20.2f}{old[n]['us_per_block']:10.2f}{new[n]['kbps']:14.1f}{old[n]['kbps']:10.1f}")

## 8. Code size and cycle counts (optional, Linux or Colab)
This installs the AVR and ARM cross-compilers and simavr, rebuilds each cipher, and counts the
cycles of 100 encryptions on a simulated ATmega328P. Skip it on Windows or macOS.

In [ ]:
HW_OUT = ""
if platform.system() == "Linux":
    need = ["avr-gcc", "arm-none-eabi-gcc"]
    if any(shutil.which(t) is None for t in need) or not os.path.exists("/usr/include/simavr"):
        sudo = [] if os.geteuid() == 0 else ["sudo"]
        subprocess.run(sudo + ["apt-get", "update", "-qq"], capture_output=True)
        subprocess.run(sudo + ["apt-get", "install", "-y", "-qq", "gcc-avr", "avr-libc", "gcc-arm-none-eabi",
                               "simavr", "libsimavr-dev", "libelf-dev"], capture_output=True)
    script = os.path.join(ROOT, "hardware", "reproduce_hardware.py")
    if os.path.exists(script):
        r = subprocess.run([sys.executable, script], cwd=os.path.join(ROOT, "hardware"), capture_output=True, text=True)
        HW_OUT = r.stdout; print(r.stdout, r.stderr[-400:])
    else:
        print("reproduce_hardware.py is not in this copy of the repository")
else:
    print("not Linux: hardware rebuild skipped")

## 9. Summary and the sentence for the paper

In [ ]:
report = dict(platform=PLATFORM, tests_pass=TESTS_OK,
              files=[dict(file=f, identical=s, different=d, timing_fields=t) for f, s, d, t in summary],
              non_timing_mismatches=mismatches, c_benchmark=C_NEW, hardware=HW_OUT)
json.dump(report, open("rerun_report.json", "w"), indent=1)
print("Tests pass:", TESTS_OK)
print("Non-timing values that differ:", len(mismatches))
print("Report written to rerun_report.json\n")
print("Sentence for Section 9.13 (replace the platform in brackets in the paper):\n")
print(f"The desktop timing figures were taken on a single desktop class platform "
      f"({PLATFORM['os']}, {PLATFORM['cpu']}, with Python {PLATFORM['python']} for the reference layer)")
print("\nIf your timings differ from Tables 7 and 8, replace those table values with the ones printed above.")